In [2]:
import random
from pathlib import Path

def split_dataset(images_dir, labels_dir, output_dir, train_ratio=0.7, val_ratio=0.2, test_ratio=0.1):
    for split in ['train', 'val', 'test']:
        Path(output_dir, 'images', split).mkdir(parents=True, exist_ok=True)
        Path(output_dir, 'labels', split).mkdir(parents=True, exist_ok=True)

    image_files = list(Path(images_dir).glob('*.jpg')) + list(Path(images_dir).glob('*.JPG'))

    valid_images = []
    for img in image_files:
        label_file = Path(labels_dir) / f"{img.stem}.txt"
        if label_file.exists():
            valid_images.append(img)
    
    print(f"Found {len(valid_images)} images with labels")

    random.shuffle(valid_images)
    train_end = int(len(valid_images) * train_ratio)
    val_end = int(len(valid_images) * (train_ratio + val_ratio))
    
    train_images = valid_images[:train_end]
    val_images = valid_images[train_end:val_end]
    test_images = valid_images[val_end:]
    
    # Copy files to respective directories
    for split_name, split_images in [('train', train_images), ('val', val_images), ('test', test_images)]:
        for img_path in split_images:
            # Copy image
            shutil.copy(img_path, Path(output_dir, 'images', split_name, img_path.name))
            
            # Copy label
            label_path = Path(labels_dir) / f"{img_path.stem}.txt"
            shutil.copy(label_path, Path(output_dir, 'labels', split_name, f"{img_path.stem}.txt"))
        
        print(f"{split_name}: {len(split_images)} images")
    
    return len(train_images), len(val_images), len(test_images)

# Run the split
output_dir = 'Cambodia_Traffic_Signs_Dataset_Split'
train_count, val_count, test_count = split_dataset(
    images_dir='Cambodia_Traffic_Signs_Datasets/images',
    labels_dir='Cambodia_Traffic_Signs_Datasets/labels',
    output_dir=output_dir,
    train_ratio=0.7,
    val_ratio=0.2,
    test_ratio=0.1
)

Found 2757 images with labels
train: 1929 images
val: 552 images
test: 276 images


In [3]:
import yaml

# Create dataset configuration for training
dataset_config = {
    'path': str(Path.cwd() / 'Cambodia_Traffic_Signs_Dataset_Split'),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'Prohibitory',
        1: 'Mandatory',
        2: 'Priority',
        3: 'Warning',
        4: 'Service',
        5: 'Other'
    }
}

# Save configuration
with open('cambodia_traffic_signs.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("Dataset configuration saved to cambodia_traffic_signs.yaml")

Dataset configuration saved to cambodia_traffic_signs.yaml


In [4]:
import glob
from collections import Counter

def analyze_dataset(dataset_dir):
    """
    Analyze the class distribution in each split
    """
    for split in ['train', 'val', 'test']:
        label_files = glob.glob(f"{dataset_dir}/labels/{split}/*.txt")
        
        class_counts = Counter()
        total_objects = 0
        
        for label_file in label_files:
            with open(label_file, 'r') as f:
                for line in f.readlines():
                    if line.strip():
                        class_id = int(line.split()[0])
                        class_counts[class_id] += 1
                        total_objects += 1
        
        print(f"\n{split.upper()} SET:")
        print(f"Images: {len(label_files)}")
        print(f"Total objects: {total_objects}")
        print("Class distribution:")
        for class_id in sorted(class_counts.keys()):
            class_name = dataset_config['names'][class_id]
            count = class_counts[class_id]
            percentage = (count / total_objects) * 100
            print(f"  Class {class_id} ({class_name}): {count} ({percentage:.1f}%)")

# Run analysis
analyze_dataset('Cambodia_Traffic_Signs_Dataset_Split')


TRAIN SET:
Images: 1929
Total objects: 2438
Class distribution:
  Class 0 (Prohibitory): 2231 (91.5%)
  Class 1 (Mandatory): 39 (1.6%)
  Class 2 (Priority): 9 (0.4%)
  Class 3 (Warning): 86 (3.5%)
  Class 4 (Service): 42 (1.7%)
  Class 5 (Other): 31 (1.3%)

VAL SET:
Images: 552
Total objects: 718
Class distribution:
  Class 0 (Prohibitory): 662 (92.2%)
  Class 1 (Mandatory): 11 (1.5%)
  Class 2 (Priority): 6 (0.8%)
  Class 3 (Warning): 19 (2.6%)
  Class 4 (Service): 11 (1.5%)
  Class 5 (Other): 9 (1.3%)

TEST SET:
Images: 276
Total objects: 339
Class distribution:
  Class 0 (Prohibitory): 313 (92.3%)
  Class 1 (Mandatory): 5 (1.5%)
  Class 2 (Priority): 3 (0.9%)
  Class 3 (Warning): 12 (3.5%)
  Class 4 (Service): 3 (0.9%)
  Class 5 (Other): 3 (0.9%)


In [7]:
from ultralytics import YOLO

# Load model
model = YOLO('yolov8n.pt')  # or yolov8s, yolov8m, etc.

# Train
results = model.train(
    data='cambodia_traffic_signs.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    name='cambodia_traffic_signs',
    save_period=10
)

Ultralytics 8.3.228 🚀 Python-3.12.7 torch-2.9.1 CPU (Apple M3 Pro)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=cambodia_traffic_signs.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=cambodia_traffic_signs, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12

KeyboardInterrupt: 